# Notebook 4 — SHAP Explanations (BCBS 239)
## Mortgage Credit Risk Modelling

This notebook visualises and interprets the SHAP output produced by `05_shap_explanations.py`.

**What SHAP tells us that AUROC cannot:**
- *Who* is risky (AUROC) vs *why* they are risky (SHAP)
- Individual loan attribution explainable to a credit committee
- Decile-level driver reports for BCBS 239 risk reporting

**Prerequisites:** Run `03_pd_ensemble.py` then `05_shap_explanations.py`.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

plt.rcParams.update({
    "figure.facecolor": "#0F1117", "axes.facecolor": "#0F1117",
    "axes.edgecolor":   "#2D3748", "axes.labelcolor": "#E2E8F0",
    "xtick.color":      "#A0AEC0", "ytick.color":     "#A0AEC0",
    "text.color":       "#E2E8F0", "grid.color":      "#1A2035",
    "legend.facecolor": "#1A2035", "legend.edgecolor":"#2D3748",
    "font.family": "monospace",    "figure.dpi": 120,
})
C = {"sky":"#38BDF8","gold":"#FBBF24","green":"#34D399","red":"#F87171","purple":"#A78BFA"}
PROC = Path("data/processed")
FIG  = Path("data/figures")
print("✓ Environment ready")

# Check outputs exist
import sys
needed = ["shap_values_oos.parquet", "shap_segment_report.csv"]
missing = [f for f in needed if not (PROC / f).exists()]
if missing:
    print("⚠  Missing files:", missing)
    print("   Run 05_shap_explanations.py first.")
else:
    print("✓ All SHAP output files found")
    sv_df  = pd.read_parquet(PROC / "shap_values_oos.parquet")
    seg_df = pd.read_csv(PROC / "shap_segment_report.csv")
    print(f"   SHAP matrix: {sv_df.shape}")
    print(f"   Segment report: {len(seg_df)} deciles")
    print(sv_df.head(3).to_string())


## 1. Global Feature Importance — Mean |SHAP|

Mean absolute SHAP value across all OOS loans, expressed in approximate probability-point (pp) units. This is the model's "average explanation" — the credit risk equivalent of a beta coefficient, but without the linearity assumption.


In [ ]:
if (PROC / "shap_values_oos.parquet").exists():
    sv_df = pd.read_parquet(PROC / "shap_values_oos.parquet")
    shap_cols = [c for c in sv_df.columns if c.startswith("shap_")]
    feat_names = [c.replace("shap_", "") for c in shap_cols]

    mean_abs = sv_df[shap_cols].abs().mean()
    mean_abs.index = feat_names
    mean_abs = mean_abs.sort_values(ascending=True)

    # Approximate conversion: log-odds → pp via σ'(logit(0.008))
    mean_abs_pp = mean_abs * 0.008 * 0.992 * 100

    LABEL_MAP = {
        "delinquency_indicator": "Current Delinquency",
        "hpi_change":            "HPI Change (Orig/Curr)",
        "occupancy_status":      "Occupancy Type",
        "orig_interest_rate":    "Note Rate (%)",
        "orig_cltv":             "Combined LTV (%)",
        "num_borrowers":         "# Borrowers",
        "credit_score":          "FICO Score",
        "property_type":         "Property Type",
        "loan_age":              "Loan Age (Months)",
        "orig_dti":              "Debt-to-Income (%)",
        "orig_upb":              "Original UPB ($)",
        "ur_3m_lag":             "Unemployment Rate (3m lag)",
    }
    labels = [LABEL_MAP.get(f, f) for f in mean_abs_pp.index]
    norm   = mean_abs_pp.values / mean_abs_pp.values.max()
    colors = plt.cm.YlOrRd(0.35 + 0.60 * norm)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(labels, mean_abs_pp.values, color=colors,
                   edgecolor="none", height=0.65)
    for bar, v in zip(bars, mean_abs_pp.values):
        ax.text(bar.get_width() + 0.0002, bar.get_y() + bar.get_height()/2,
                f"+{v:.4f} pp", va="center", fontsize=9, color="#E2E8F0")

    ax.set_xlabel("Mean |SHAP| — approx. impact on default probability (pp)", fontsize=10)
    ax.set_title("SHAP Global Feature Importance — XGBoost PD Model (OOS)\n"
                 "Each bar = average absolute contribution across all held-out loans",
                 fontsize=11, fontweight="bold", color="white", pad=12)
    ax.grid(True, axis="x", alpha=0.25)
    ax.set_xlim(0, mean_abs_pp.max() * 1.28)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()
    print(f"\nTop driver: {mean_abs_pp.idxmax()} ({mean_abs_pp.max():.4f} pp average impact)")


## 2. Beeswarm Plot — Feature Impact Distribution

Each dot is one loan. The x-position is its SHAP value for that feature (positive = raises PD, negative = lowers PD). Colour encodes the raw feature value (red = high, blue = low).

This is the most information-dense SHAP visualisation: it shows direction, magnitude, *and* distribution simultaneously.


In [ ]:
if (PROC / "shap_values_oos.parquet").exists():
    shap_cols = [c for c in sv_df.columns if c.startswith("shap_")]
    feat_names = [c.replace("shap_", "") for c in shap_cols]
    shap_vals  = sv_df[shap_cols].values

    # Reconstruct raw feature matrix from SHAP file if orig feature cols available
    # (shap_values_oos contains shap_ cols + log_odds + predicted_pd + actual_default)
    non_shap = [c for c in sv_df.columns if not c.startswith("shap_")]

    mean_abs = np.abs(shap_vals).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[::-1][:10]

    fig, ax = plt.subplots(figsize=(11, 7))
    for row_i, fi in enumerate(top_idx[::-1]):
        sv_row = shap_vals[:, fi]
        # Use rank of shap value as proxy for raw feature value (direction preserved)
        rank_norm = (np.argsort(np.argsort(sv_row)) / len(sv_row)).clip(0, 1)
        jitter = np.random.default_rng(row_i).uniform(-0.3, 0.3, len(sv_row))
        sc = ax.scatter(sv_row,
                        np.full(len(sv_row), row_i) + jitter,
                        c=rank_norm, cmap="coolwarm",
                        s=5, alpha=0.45, linewidths=0, vmin=0, vmax=1)

    ax.axvline(0, color="#4B5563", linewidth=1.3, linestyle="--")
    labels_top = [LABEL_MAP.get(feat_names[i], feat_names[i]) for i in top_idx[::-1]]
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels(labels_top, fontsize=10)
    ax.set_xlabel("SHAP Value — log-odds impact on default probability", fontsize=10)
    ax.set_title("SHAP Beeswarm — Top 10 Features  (OOS set)\n"
                 "Each point = one loan  ·  Red = high feature value  ·  Blue = low",
                 fontsize=11, fontweight="bold", color="white", pad=12)
    cbar = fig.colorbar(sc, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("Feature value\n(normalised)", fontsize=9, color="#CBD5E1")
    ax.grid(True, axis="x", alpha=0.2)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()


## 3. Segment Report — BCBS 239 Risk Attribution by Decile

Banks must be able to aggregate and report risk drivers across portfolio segments. This table shows the top three SHAP drivers per PD decile — directly satisfying BCBS 239 Principle 6 (Completeness).


In [ ]:
if (PROC / "shap_segment_report.csv").exists():
    seg = pd.read_csv(PROC / "shap_segment_report.csv")
    print("SHAP Segment Report — Portfolio Deciles Ranked by Predicted PD")
    print("=" * 80)
    print(seg.to_string(index=False))
    print()

    # Visualise predicted vs observed default rate by decile
    fig, ax = plt.subplots(figsize=(10, 4))
    x = seg["decile"].values
    ax.bar(x - 0.2, seg["mean_pd_pct"],     width=0.38, color=C["sky"],
           alpha=0.85, label="Mean predicted PD (%)")
    ax.bar(x + 0.2, seg["observed_dr_pct"], width=0.38, color=C["red"],
           alpha=0.85, label="Observed default rate (%)")
    ax.set_xlabel("Score Decile (1 = lowest risk, 10 = highest risk)", fontsize=10)
    ax.set_ylabel("Rate (%)", fontsize=10)
    ax.set_title("Predicted PD vs Observed Default Rate by Risk Decile\n"
                 "BCBS 239: Accuracy and integrity of risk reports",
                 fontsize=11, fontweight="bold", color="white", pad=12)
    ax.legend(fontsize=9)
    ax.grid(True, axis="y", alpha=0.25)
    ax.set_xticks(x)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()


## 4. Pre-generated SHAP Figures

The following figures were generated by `05_shap_explanations.py` and saved to `data/figures/`.


In [ ]:
from IPython.display import Image, display
import os

shap_figs = [
    ("shap_global_importance.png",    "Global feature importance (mean |SHAP|)"),
    ("shap_beeswarm.png",             "Beeswarm — distribution of SHAP values per feature"),
    ("shap_waterfall_high_risk.png",  "Waterfall — highest-risk loan (individual attribution)"),
    ("shap_waterfall_low_risk.png",   "Waterfall — lowest-risk loan (individual attribution)"),
    ("shap_dependence_credit_score.png","Dependence plot — FICO score × interaction feature"),
    ("shap_segment_report.png",       "Segment report — top SHAP drivers by risk decile"),
]

for fname, caption in shap_figs:
    path = FIG / fname
    if path.exists():
        print(f"\n{'─'*60}")
        print(f"  {caption}")
        print(f"{'─'*60}")
        display(Image(str(path), width=800))
    else:
        print(f"  ⚠  {fname} not found — run 05_shap_explanations.py first")


## Summary

| SHAP Output | Regulatory Use |
|---|---|
| Global importance | Model governance — which features drive the model |
| Beeswarm | Directional consistency check — does FICO score behave as expected? |
| Waterfall | Individual decision justification — SR 11-7 model risk |
| Segment report | Portfolio risk aggregation — BCBS 239 Principles 6 & 11 |

The waterfall plots are the key deliverable: they allow a credit officer to explain *any individual loan's PD* to an underwriter, risk committee, or regulator in plain terms.
